# A1 · 2022/23 DFS events: what is the data?

Step 1: just look at the raw DFS event files NESO publishes for 2022/23.

The first code cell downloads any raw data that's missing (`data/` is not committed), so
this notebook runs from a fresh clone. Loaders: `uk_dfs.data.neso` (DFS events, national
demand) and `uk_dfs.data.elexon` (5-minute generation).

In [ ]:
import pandas as pd

from uk_dfs.config import RAW_DIR
from uk_dfs.data import elexon, neso

# Download any raw data that isn't on disk yet (first run: ~1 minute; after that: instant)
neso.fetch_dfs()
neso.fetch_demand()
elexon.fetch_fuelinst("2022-11", "2023-03")  # 5-minute data for the 2022/23 season

DFS = RAW_DIR / "neso_dfs"

## Live events
One row per half-hour of each event.

In [ ]:
live = pd.read_csv(DFS / "2223_live_summary.csv")
live

## Test events
Same columns. In 2022/23 most events were "tests" run to build up participation.

In [ ]:
test = pd.read_csv(DFS / "2223_test_summary.csv")
test

## 5-minute generation (Elexon FUELINST)
One Parquet file per month in `data/raw/elexon_fuelinst/`. "Long" format: one row per
5-minute interval × fuel type, with generation in MW. Interconnectors (`INT…`) and pumped
storage (`PS`) go negative when GB is exporting or pumping.

In [ ]:
fuel = pd.read_parquet(RAW_DIR / "elexon_fuelinst" / "2023-01.parquet")
print(fuel.shape)
print(fuel.fuelType.unique())
fuel.head(20)

### One day, one row per 5 minutes
Reshape to "wide": one row per 5-minute interval, one column per fuel type, plus a total.
24 Jan 2023 is the second live event (16:30–18:00). Times are UTC, which in January is
the same as UK clock time.

In [ ]:
day = fuel[fuel.startTime.dt.date == pd.Timestamp("2023-01-24").date()]
day = day.pivot(index="startTime", columns="fuelType", values="generation")
day["TOTAL"] = day.sum(axis=1)
day